# CSV 驱动的全景 SAR 特征提取（v8）

## 当前算法与运行方式

- CSV 为唯一用户数据输入：site_id、分号分隔 scene_ids；不需要 MPA 面资产。
- 处理原景 SAR footprint，不把 SAR 独立裁成空间小块。site_id 仅作元数据。
- MODIS/006/MOD44W（水体=1）与 JRC occurrence≥10% 组合剔陆；近岸分类精度受其原生分辨率限制。
- SAR/Lee/阈值/形态学/纹理和几何统计为 10 m；输出每行是 SAR 原景投影下的 6 km 网格（边界格取 footprint 内的部分）。
- 两次窗口均值改用 coveringGrid + reduceRegions(scale=10)，并用格中心点回写粗均值图，继续双线性回采样阈值场。**不再使用 reduceResolution/100 m 中间层**：旧链在真实全景已复现 Reprojection output too large。这是计算接口改变，不是把 SAR 降成 6 km。
- 保留亮像元替换、阈值、形态学、0.02 km² 几何面积筛选、多暗斑特征、60 m 几何背景环和 16 项特征。Lee 从 SAR 自身开始进行等价功率转换；CRS 显式传递，纹理每景构建一次，网格内矢量结果复用，空暗斑分支安全返回。
- 输出仅 CSV：site_id、scene_id、sar_time + 16 项特征。相同景在多站点重复出现会重复计算全景，并附不同站点标签；不是站点多边形内的统计。

## 验证与尚存资源限制

独立限时的合成服务端查询通过：10 m 投影、两次均值、900 像元暗斑分割、1 个暗斑的 16 项特征、空暗斑分支。多格测试中新旧粗均值差约 1e-7 dB。真实整景同步均值查询超过 60 秒，**未证明完整全景导出成功**；不再把整景 getInfo 当快速诊断。

全景几何面积筛选仍需要整景 10 m 矢量化，可能触发内存/复杂几何/耗时限额。提高 maxPixels 或 tileScale 不能保证所有景一次成功。为保留跨网格暗斑面积筛选语义，未替换成近似像元计数，也未切块。必须观察异步导出的最终状态，不能仅以提交成功认定计算完成。

## 测试范围与执行顺序

CSV 首个有效站点 AIAN18，共 14 景；**默认每次仅新增提交 1 个任务**。先确认首任务完成再继续。执行顺序：EE 初始化 → 核心函数 → 读取首站点 → 提交函数定义 → 短检查 → 手动提交 → 状态查询。

新任务名前缀 test_fullscene_v8_。旧任务计算图不会随代码更新；内核重启后任务字典清空，提交前须检查现有 GEE Tasks 避免重复。移除了阻塞内核的合成大查询，未自动提交任何新任务。

真实首景的 9 个相邻原生网格查询通过：格中心回采样值与本格均值一致，10 m/6 km 投影正确。第二次均值属性直接复用，避免为输出格额外执行逐格均值图查询。完整全景异步导出仍待用户手动验证。


In [7]:
#### CSV 驱动的全景水域 SAR 分割与 6 km 特征提取 ####
import ee
ee.Authenticate()
ee.Initialize(project='ee-zns0917')

import os
os.environ['KMP_DUPLICATE_LIB_OK']='TRUE'

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [8]:
import pandas as pd

# ----------------- 核心参数配置：与 Seg_Square 保持一致 -----------------
LEE_KERNEL_RADIUS = 1
LEE_ENL = 4
ABS_THRESHOLD_DB = -21
DB_THRESHOLD_SHIFT = 3
BRIGHT_THRESHOLD = -15
MORPH_RADIUS = 1
MIN_AREA_KM2 = 0.02
MIN_OBJECT_PIXELS = 9
SCENE_VECTOR_MAX_PIXELS = 1_000_000_000  # 全景 10 m 矢量化上限，不改变像元尺度
FEATURE_SCALE = 10
GRID_SCALE = 6000       # 保留 MPA 单景矩阵降采样框架
GRID_REDUCTION_MAX_PIXELS = 1_000_000  # 每个 6 km 网格的 10 m 统计上限
TEXTURE_SCALE = 10
TEXTURE_LEVELS = 32
BACKGROUND_RING_M = 60
EPS = 1e-6
PI = 3.141592653589793
SAFE_FILL = -999

FEATURE_COLUMNS = [
    'dark_spot_area_mean_m2', 'vv_filtered_stdDev', 'dark_spot_count',
    'sar_hour', 'corr', 'local_mean_after', 'wind_mean',
    'dark_spot_compactness_area_weighted', 'intensity_ratio',
    'wind_dir_sin', 'asm', 'dark_spot_area_std_m2',
    'dark_spot_lwr_mean', 'isdr', 'isri', 'shade'
]

# 唯一用户输入为 CSV；site_id 仅用作元数据，不再加载 MPA 矢量资产。
SAR_info_csv = 'F:/MYR/Research_/ZHANG/Oiling/results/sar_2015_protected_areas_with_sceneid.csv'
output_feature = 'test_MPA_Features_2015'

# 自动海陆筛选：MOD44W 保留海洋，JRC 补充已识别地表水。
# JRC occurrence 单独使用不覆盖大部分公海；无数据不能直接视为海洋。
JRC_WATER_OCCURRENCE_MIN = 10
LAND_WATER_REFERENCE_YEAR = 2015


def get_water_mask():
    """全球海陆分类 + JRC 水体；不依赖任何用户上传的面资产。"""
    modis_collection = (ee.ImageCollection('MODIS/006/MOD44W')
                        .filterDate(f'{LAND_WATER_REFERENCE_YEAR}-01-01',
                                    f'{LAND_WATER_REFERENCE_YEAR + 1}-01-01')
                        .select('water_mask')
                        .map(lambda image: ee.Image(image).toByte()))
    # 先放入全陆地兜底影像，再叠加 MOD44W；即使集合为空也不会对 null 调用 select。
    modis_fallback = ee.Image.constant(0).rename('water_mask').toByte()
    modis_water = (ee.ImageCollection([modis_fallback])
                   .merge(modis_collection).mosaic().eq(1))
    occurrence = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence')
    # where 输出使用常数影像的完整二值 mask，缺失 occurrence 不推断为海洋。
    jrc_water = ee.Image.constant(0).where(occurrence.gte(JRC_WATER_OCCURRENCE_MIN), 1)
    return modis_water.Or(jrc_water).rename('water_mask')


def validate_water_mask_source():
    """提交导出前验证海陆掩膜，空集合不再拖到服务端任务才报错。"""
    collection = ee.ImageCollection('MODIS/006/MOD44W').filterDate(
        f'{LAND_WATER_REFERENCE_YEAR}-01-01', f'{LAND_WATER_REFERENCE_YEAR + 1}-01-01'
    )
    count = collection.size().getInfo()
    if count == 0:
        raise ValueError(f'MODIS/006/MOD44W 在 {LAND_WATER_REFERENCE_YEAR} 年没有影像；未提交任务。')
    bands = ee.Image(collection.first()).bandNames().getInfo()
    if 'water_mask' not in bands:
        raise ValueError(f'海陆掩膜缺少 water_mask 波段：{bands}')
    # 实际计算一小块组合掩膜，提前发现 mosaic 的波段类型冲突。
    mask_probe = get_water_mask().reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=ee.Geometry.Point([0, 0]).buffer(500),
        scale=250, maxPixels=256
    ).getInfo()
    if mask_probe.get('water_mask') is None:
        raise ValueError(f'组合水体掩膜检查返回空值：{mask_probe}')
    return {'collection': 'MODIS/006/MOD44W', 'year': LAND_WATER_REFERENCE_YEAR, 'count': count, 'bands': bands, 'mask_probe': mask_probe}


def _safe_num(x, default=SAFE_FILL):
    return ee.Number(ee.List([x, default]).reduce(ee.Reducer.firstNonNull()))


def _safe_div(a, b, default=SAFE_FILL):
    a = ee.Number(a)
    b = ee.Number(b)
    return ee.Number(ee.Algorithms.If(b.abs().gt(EPS), a.divide(b), default))


def lee_filter(img_db):
    """与 Seg_Square 一致：功率域局部统计 MMSE Lee 滤波，返回 dB。"""
    img_db = img_db.toFloat()
    # exp(dB × ln(10)/10) = 10**(dB/10)：同一功率转换公式。
    # 从 SAR 影像本身开始，保留原生投影、footprint 和 mask，不继承常数影像的全球范围。
    img_power = img_db.divide(10).multiply(2.302585092994046).exp()
    kernel = ee.Kernel.square(radius=LEE_KERNEL_RADIUS, units='pixels')
    mean_power = img_power.reduceNeighborhood(ee.Reducer.mean(), kernel)
    mean_sq = img_power.pow(2).reduceNeighborhood(ee.Reducer.mean(), kernel)
    var_power = mean_sq.subtract(mean_power.pow(2)).max(0)
    noise_var = mean_power.pow(2).divide(LEE_ENL)
    signal_var = var_power.subtract(noise_var).max(0)
    weights = signal_var.divide(var_power.max(1e-12)).clamp(0, 1)
    filtered_power = mean_power.add(weights.multiply(img_power.subtract(mean_power))).max(1e-12)
    return filtered_power.log10().multiply(10).rename('VV')


def get_era5_wind_image(sar_time, region):
    """匹配 SAR 成像小时的 ERA5 风场，返回风速及风向 sin 分量。"""
    hour_start = ee.Date.fromYMD(
        sar_time.get('year'), sar_time.get('month'), sar_time.get('day')
    ).advance(sar_time.get('hour'), 'hour')
    hour_end = hour_start.advance(1, 'hour')
    era5_collection = (ee.ImageCollection('ECMWF/ERA5/HOURLY')
                       .filterBounds(region)
                       .filterDate(hour_start, hour_end)
                       .select(['u_component_of_wind_10m', 'v_component_of_wind_10m']))
    # 有数据时必须取真实 ERA5，不能把兜底放首位后直接 first()。
    # 缺失时使用全掩膜波段，输出缺失值，而不是伪造零风速。
    era5_fallback = (ee.Image.constant([0, 0])
                     .rename(['u_component_of_wind_10m', 'v_component_of_wind_10m'])
                     .updateMask(ee.Image.constant(0)))
    era5 = ee.Image(ee.Algorithms.If(
        era5_collection.size().gt(0), era5_collection.first(), era5_fallback
    ))
    u_img = era5.select('u_component_of_wind_10m')
    v_img = era5.select('v_component_of_wind_10m')
    speed_img = u_img.pow(2).add(v_img.pow(2)).sqrt().rename('wind_speed')
    wind_dir_sin_img = v_img.divide(speed_img.max(1e-6)).rename('wind_dir_sin')
    return speed_img.addBands(wind_dir_sin_img)


def _coarse_water_mean(img, band_name, region=None, grid=None, return_stats=False):
    """按原景 6 km 网格对 10 m 水域像元求均值，避免强制生成全景细栅格。"""
    native_proj = img.projection()
    coarse_proj = native_proj.atScale(GRID_SCALE)
    region = img.geometry() if region is None else region
    grid = region.coveringGrid(coarse_proj) if grid is None else grid
    # 单格 600×600=360000 像元，上限留边界余量；没有 bestEffort 降分辨率。
    # 单输入 reduceRegions 的字段显式命名为 VV，避免读取错误字段。
    stats = img.reduceRegions(
        collection=grid, reducer=ee.Reducer.mean().setOutputs(['VV']),
        crs=native_proj, scale=FEATURE_SCALE, tileScale=8,
        maxPixelsPerRegion=GRID_REDUCTION_MAX_PIXELS
    ).filter(ee.Filter.notNull(['VV']))
    # 矩形直接 reduceToImage 会在公共边界混入邻格属性；只在格中心放均值。
    centers = stats.map(lambda f: ee.Feature(
        ee.Feature(f).geometry().centroid(1, coarse_proj), {'VV': ee.Feature(f).get('VV')}
    ))
    # 此处唯一强制重投影的对象是稀疏格中心属性，输出为粗网格（不是 10 m SAR）。
    coarse_image = centers.reduceToImage(['VV'], ee.Reducer.first()).rename(band_name).reproject(coarse_proj)
    if return_stats:
        grid_stats = stats.map(lambda f: ee.Feature(f).set(band_name, ee.Feature(f).get('VV')))
        return coarse_image, grid_stats
    return coarse_image


def matrix_pooling_segmentation(sar_filtered, region, grid=None, return_stats=False):
    """景内 6 km 矩阵降采样两步阈值分割；阈值及形态学与 Seg_Square 一致。"""
    mean1_coarse = _coarse_water_mean(sar_filtered, 'local_mean_before', region, grid)
    mean1_10m = mean1_coarse.resample('bilinear')

    # 与 Square 一致：亮像元定义为 > -15 dB，替换为第一次局部均值。
    bright_mask = sar_filtered.gt(BRIGHT_THRESHOLD)
    img_suppressed = sar_filtered.where(bright_mask, mean1_10m).rename('VV')
    mean2_coarse, mean2_stats = _coarse_water_mean(
        img_suppressed, 'local_mean_after', region, grid, return_stats=True
    )
    mean2_10m = mean2_coarse.resample('bilinear')

    condition1 = img_suppressed.lt(mean2_10m.subtract(DB_THRESHOLD_SHIFT))
    condition2 = img_suppressed.lt(ABS_THRESHOLD_DB)
    oil_mask_raw = condition1.And(condition2).rename('oil_mask')
    kernel = ee.Kernel.square(MORPH_RADIUS)
    oil_mask_morph = oil_mask_raw.focal_max(kernel=kernel).focal_min(kernel=kernel)
    # 形态学不能把油膜扩回陆地或 SAR 无效像元。
    oil_mask_morph = oil_mask_morph.updateMask(sar_filtered.mask())

    # 与 Square 一致：矢量化连通斑块并剔除面积小于 0.02 km² 的斑块。
    patch_vectors = oil_mask_morph.selfMask().reduceToVectors(
        geometry=region, crs=sar_filtered.projection(), scale=FEATURE_SCALE, geometryType='polygon',
        eightConnected=True, labelProperty='label',
        maxPixels=SCENE_VECTOR_MAX_PIXELS, bestEffort=False, tileScale=8
    )
    patch_vectors = patch_vectors.map(
        lambda f: ee.Feature(f).set(
            'patch_area_km2', ee.Feature(f).geometry().area(1).divide(1e6)
        )
    )
    big_patches = patch_vectors.filter(ee.Filter.gte('patch_area_km2', MIN_AREA_KM2))
    oil_mask_final = (sar_filtered.multiply(0).unmask(0).byte()
                      .paint(big_patches, 1)
                      .And(oil_mask_morph).unmask(0).rename('oil_mask')
                      .setDefaultProjection(sar_filtered.projection()).clip(region))
    if return_stats:
        return oil_mask_final, img_suppressed, mean2_coarse, mean2_stats
    return oil_mask_final, img_suppressed, mean2_coarse


def _multi_dark_spot_features(patches):
    """复用当前网格已经矢量化的斑块，汇总面积、数量、紧凑度和长宽比。"""
    def add_patch_geometry(feature):
        geom = ee.Feature(feature).geometry()
        area_m2 = geom.area(1)
        perimeter_m = geom.perimeter(1)
        centroid = geom.centroid(1).coordinates()
        latitude = ee.Number(centroid.get(1))
        bounds = ee.List(geom.bounds(1).coordinates().get(0))
        p0, p1, p2 = ee.List(bounds.get(0)), ee.List(bounds.get(1)), ee.List(bounds.get(2))
        # 经度方向距离按中心纬度修正，不使用与纬度无关的固定经度换算。
        width_m = (ee.Number(p1.get(0)).subtract(ee.Number(p0.get(0))).abs()
                   .multiply(111320).multiply(latitude.multiply(PI / 180).cos().abs()))
        height_m = ee.Number(p2.get(1)).subtract(ee.Number(p1.get(1))).abs().multiply(111320)
        lwr = _safe_div(width_m.max(height_m), width_m.min(height_m), 0)
        compactness = _safe_div(
            ee.Number(4).multiply(PI).multiply(area_m2), perimeter_m.pow(2), 0
        )
        return ee.Feature(feature).set({
            'patch_area_m2': area_m2,
            'patch_lwr': lwr,
            'patch_compactness': compactness
        })

    patches = patches.map(add_patch_geometry)
    area_stats = ee.Dictionary(patches.aggregate_stats('patch_area_m2'))
    lwr_stats = ee.Dictionary(patches.aggregate_stats('patch_lwr'))
    total_area = _safe_num(patches.aggregate_sum('patch_area_m2'), 0)
    weighted = patches.map(lambda f: ee.Feature(f).set(
        'compactness_x_area',
        ee.Number(ee.Feature(f).get('patch_compactness'))
        .multiply(ee.Number(ee.Feature(f).get('patch_area_m2')))
    ))
    return ee.Dictionary({
        'dark_spot_area_mean_m2': _safe_num(area_stats.get('mean'), 0),
        'dark_spot_count': patches.size(),
        'dark_spot_compactness_area_weighted': _safe_div(
            _safe_num(weighted.aggregate_sum('compactness_x_area'), 0), total_area, 0
        ),
        'dark_spot_area_std_m2': _safe_num(area_stats.get('sample_sd'), 0),
        'dark_spot_lwr_mean': _safe_num(lwr_stats.get('mean'), 0)
    })


def _grid_feature(grid, oil_mask, raw_img, filtered_img, mean2_coarse,
                  wind_img, texture_img, sar_time, scene_id, site_id):
    """原生 6 km 网格；空暗斑分支不触发矢量化、缓冲和对象纹理统计。"""
    grid = ee.Feature(grid)
    region = grid.geometry()
    sar_proj = raw_img.projection()
    pixel_count = _safe_num(oil_mask.eq(1).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=region, crs=sar_proj, scale=FEATURE_SCALE,
        maxPixels=1e8, tileScale=8
    ).get('oil_mask'), 0)
    vv_stats = filtered_img.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
        geometry=region, crs=sar_proj, scale=FEATURE_SCALE,
        maxPixels=1e8, tileScale=8
    )
    wind_stats = wind_img.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=region, scale=1000, maxPixels=1e6
    )
    # 直接读取本格第二次均值，避免边界格中心不在 footprint 时返回空值。

    window_dict = ee.Dictionary({
        'vv_filtered_stdDev': _safe_num(vv_stats.get('VV_stdDev')),
        'sar_hour': sar_time.get('hour'),
        'local_mean_after': _safe_num(grid.get('local_mean_after')),
        'wind_mean': _safe_num(wind_stats.get('wind_speed')),
        'wind_dir_sin': _safe_num(wind_stats.get('wind_dir_sin'), 0)
    })

    def when_valid_object():
        obj = oil_mask.eq(1).selfMask().clip(region)
        patches = oil_mask.selfMask().reduceToVectors(
            geometry=region, crs=sar_proj, scale=FEATURE_SCALE,
            geometryType='polygon', eightConnected=True, labelProperty='label',
            bestEffort=False, maxPixels=1e8, tileScale=8
        )
        # 本格只矢量化一次，形状统计与 60 m 几何背景环复用它。
        dark_spot_dict = _multi_dark_spot_features(patches)
        obj_geom = patches.geometry(1)
        outer_ring = obj_geom.buffer(BACKGROUND_RING_M).difference(obj_geom, 1)
        background_mask = oil_mask.eq(1).unmask(0).Not()
        obj_stats = raw_img.updateMask(obj).reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
            geometry=region, crs=sar_proj, scale=FEATURE_SCALE,
            maxPixels=1e8, tileScale=8
        )
        bg_stats = raw_img.updateMask(background_mask).clip(region).reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
            geometry=outer_ring, crs=sar_proj, scale=FEATURE_SCALE,
            maxPixels=1e8, tileScale=8
        )
        fallback_bg_stats = raw_img.updateMask(background_mask).reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
            geometry=region, crs=sar_proj, scale=FEATURE_SCALE,
            maxPixels=1e8, tileScale=8
        )
        mu_sce = _safe_num(ee.List([
            bg_stats.get('VV_mean'), fallback_bg_stats.get('VV_mean')
        ]).reduce(ee.Reducer.firstNonNull()))
        sigma_sce = _safe_num(ee.List([
            bg_stats.get('VV_stdDev'), fallback_bg_stats.get('VV_stdDev')
        ]).reduce(ee.Reducer.firstNonNull()))
        mu_obj = _safe_num(obj_stats.get('VV_mean'), 0)
        sigma_obj = _safe_num(obj_stats.get('VV_stdDev'), 0)
        tex_stats = texture_img.updateMask(obj).reduceRegion(
            reducer=ee.Reducer.mean(), geometry=region, crs=sar_proj, scale=TEXTURE_SCALE,
            maxPixels=1e8, tileScale=8
        )
        return dark_spot_dict.combine(ee.Dictionary({
            'corr': _safe_num(tex_stats.get('VV_corr')),
            'asm': _safe_num(tex_stats.get('VV_asm')),
            'shade': _safe_num(tex_stats.get('VV_shade')),
            'intensity_ratio': _safe_div(mu_obj, mu_sce, 0),
            'isdr': _safe_div(sigma_obj, sigma_sce, 0),
            'isri': _safe_div(mu_obj, sigma_obj, 0)
        }))

    empty_defaults = ee.Dictionary({
        'dark_spot_area_mean_m2': 0, 'dark_spot_count': 0,
        'dark_spot_compactness_area_weighted': 0, 'dark_spot_area_std_m2': 0,
        'dark_spot_lwr_mean': 0, 'corr': SAFE_FILL, 'asm': SAFE_FILL,
        'shade': SAFE_FILL, 'intensity_ratio': 0, 'isdr': 0, 'isri': 0
    })
    object_dict = ee.Dictionary(ee.Algorithms.If(
        pixel_count.gte(MIN_OBJECT_PIXELS), when_valid_object(), empty_defaults
    ))
    return ee.Feature(region, window_dict.combine(object_dict)).set({
        'site_id': site_id, 'scene_id': scene_id,
        'sar_time': sar_time.format('YYYY-MM-dd HH:mm:ss')
    })


def process_single_scene(scene_id, site_id):
    """CSV 驱动全景处理；site_id 仅写入每个网格的元数据，不用于裁剪。"""
    scene_id = str(scene_id).strip()
    site_id = str(site_id).strip()
    image_id = (scene_id if scene_id.startswith('COPERNICUS/S1_GRD/')
                else f'COPERNICUS/S1_GRD/{scene_id}')
    s1_image = ee.Image(image_id)
    sar_time = ee.Date(s1_image.get('system:time_start'))
    scene_geom = s1_image.geometry()
    sar_db = s1_image.select('VV').rename('VV')
    # 先剔除陆地，再进行 Lee 统计、两次粗网格均值及所有后续特征统计。
    raw_img = sar_db.updateMask(get_water_mask()).clip(scene_geom)
    # 显式保留原景 VV 投影，避免常数影像参与运算后落到默认投影。
    sar_filtered = lee_filter(raw_img).setDefaultProjection(sar_db.projection())

    coarse_proj = sar_db.projection().atScale(GRID_SCALE)
    # 完整方格用于均值统计；SAR 没有被分块裁剪，跨格滤波与形态学仍连续。
    native_grid = scene_geom.coveringGrid(coarse_proj)
    oil_mask, img_suppressed, mean2_coarse, valid_grid = matrix_pooling_segmentation(
        sar_filtered, scene_geom, native_grid, return_stats=True
    )
    wind_img = get_era5_wind_image(sar_time, scene_geom)
    # 单景只定义一份 GLCM；不先裁剪网格，保持纹理邻域连续。
    texture_base = (raw_img.clamp(-35, 0).unitScale(-35, 0)
                    .multiply(TEXTURE_LEVELS - 1).toInt().rename('VV'))
    texture_img = texture_base.glcmTexture(size=1).select(['VV_corr', 'VV_asm', 'VV_shade'])

    # 复用第二次均值统计的网格属性，不再对均值图逐格 reduceRegion。
    # 边界格统计限定到 footprint；整景图像及其跨格邻域不做独立裁剪。
    grid_polys = valid_grid.map(lambda f: ee.Feature(
        ee.Feature(f).geometry().intersection(scene_geom, 1),
        {'local_mean_after': ee.Feature(f).get('local_mean_after')}
    ))
    return grid_polys.map(lambda grid: _grid_feature(
        grid, oil_mask, raw_img, sar_filtered, mean2_coarse,
        wind_img, texture_img, sar_time, scene_id, site_id
    ))


In [9]:
import csv
from pathlib import Path

# 独立读取源笔记所用 CSV；不触发 EE 查询或导出。
TEST_CSV = Path('F:/MYR/Research_/ZHANG/Oiling/results/sar_2015_protected_areas_with_sceneid.csv')
with TEST_CSV.open(encoding='utf-8-sig', newline='') as handle:
    reader = csv.DictReader(handle)
    required_columns = {'site_id', 'scene_ids'}
    if not required_columns.issubset(reader.fieldnames or []):
        raise ValueError(f'CSV 缺少字段：{required_columns - set(reader.fieldnames or [])}')
    first_site_row = next((
        row for row in reader
        if (row.get('has_image', '1').strip() in {'1', '1.0'}
            and row['site_id'].strip() and row['scene_ids'].strip())
    ), None)

if first_site_row is None:
    raise ValueError('CSV 中没有包含 SAR 景的有效站点。')

test_site_id = first_site_row['site_id'].strip()
test_scene_ids = list(dict.fromkeys(
    scene.strip() for scene in first_site_row['scene_ids'].split(';') if scene.strip()
))
{'site_id': test_site_id, 'scene_count': len(test_scene_ids), 'scene_ids': test_scene_ids}

{'site_id': 'AIAN18',
 'scene_count': 14,
 'scene_ids': ['S1A_IW_GRDH_1SDV_20150511T224246_20150511T224308_005880_007928_C66D',
  'S1A_IW_GRDH_1SSV_20150808T102414_20150808T102443_007170_009CAC_F74A',
  'S1A_IW_GRDH_1SSV_20150815T224250_20150815T224307_007280_009FA1_C78A',
  'S1A_IW_GRDH_1SSV_20150901T102416_20150901T102444_007520_00A639_6898',
  'S1A_IW_GRDH_1SSV_20150908T224251_20150908T224308_007630_00A929_2216',
  'S1A_IW_GRDH_1SSV_20150925T102416_20150925T102445_007870_00AF98_615C',
  'S1A_IW_GRDH_1SSV_20151002T224252_20151002T224309_007980_00B28C_1616',
  'S1A_IW_GRDH_1SSV_20151019T102417_20151019T102445_008220_00B914_89B9',
  'S1A_IW_GRDH_1SSV_20151026T224252_20151026T224309_008330_00BBFD_F420',
  'S1A_IW_GRDH_1SSV_20151112T102355_20151112T102420_008570_00C269_110A',
  'S1A_IW_GRDH_1SDV_20151119T224247_20151119T224312_008680_00C582_CB97',
  'S1A_IW_GRDH_1SSV_20151206T102355_20151206T102420_008920_00CC4A_AC2F',
  'S1A_IW_GRDH_1SSV_20151213T224245_20151213T224302_009030_00CF41_CA6

In [10]:
# 同一内核内保留任务记录；失败/取消可重试，排队/运行/完成的任务不重复提交。
if 'test_full_scene_tasks' not in globals():
    test_full_scene_tasks = {}
if 'test_full_scene_submission_errors' not in globals():
    test_full_scene_submission_errors = {}


def submit_first_site_sar(max_new_tasks=1):
    """仅提交首站点全部 SAR 景；提交前检查水体数据源和每景 ERA5。"""
    if not isinstance(max_new_tasks, int) or max_new_tasks < 1:
        raise ValueError('max_new_tasks 必须为正整数。')
    validate_water_mask_source()
    submitted_this_run = 0

    for scene_idx, scene_id in enumerate(test_scene_ids, start=1):
        if submitted_this_run >= max_new_tasks:
            break
        image_id = (
            scene_id if scene_id.startswith('COPERNICUS/S1_GRD/')
            else f'COPERNICUS/S1_GRD/{scene_id}'
        )
        try:
            if scene_id in test_full_scene_tasks:
                previous_state = test_full_scene_tasks[scene_id].status().get('state')
                if previous_state not in {'FAILED', 'CANCELLED'}:
                    print(f'[{scene_idx}/{len(test_scene_ids)}] {previous_state}，跳过：{scene_id}', flush=True)
                    continue
                print(f'[{scene_idx}/{len(test_scene_ids)}] 重试 {previous_state} 任务：{scene_id}', flush=True)

            # 小查询检查影像和小时风场，不下载全景分割结果。
            print(f'[{scene_idx}/{len(test_scene_ids)}] 检查影像：{image_id}', flush=True)
            image = ee.Image(image_id)
            bands = image.bandNames().getInfo()
            if 'VV' not in bands:
                raise ValueError(f'影像不包含 VV 波段：{bands}')
            sar_time = ee.Date(image.get('system:time_start'))
            hour_start = ee.Date.fromYMD(
                sar_time.get('year'), sar_time.get('month'), sar_time.get('day')
            ).advance(sar_time.get('hour'), 'hour')
            era5_count = (ee.ImageCollection('ECMWF/ERA5/HOURLY')
                          .filterBounds(image.geometry())
                          .filterDate(hour_start, hour_start.advance(1, 'hour'))
                          .size().getInfo())
            if era5_count == 0:
                raise ValueError(f'ERA5 对应小时没有影像：{scene_id}；未提交本景任务。')

            scene_fc = process_single_scene(scene_id, test_site_id)
            short_id = scene_id.rsplit('/', 1)[-1]
            date_part = short_id.split('_')[4][:8]
            scene_suffix = '_'.join(short_id.split('_')[-2:])
            task_desc = f'test_fullscene_v8_{test_site_id}_{scene_idx:02d}_{date_part}_{scene_suffix}'
            task = ee.batch.Export.table.toDrive(
                collection=scene_fc,
                description=task_desc,
                folder=output_feature,
                fileNamePrefix=task_desc,
                fileFormat='CSV',
                selectors=['site_id', 'scene_id', 'sar_time'] + FEATURE_COLUMNS
            )
            task.start()
            submitted_this_run += 1
            test_full_scene_tasks[scene_id] = task
            test_full_scene_submission_errors.pop(scene_id, None)
            print(f'已提交：{task_desc}；task_id={task.id}', flush=True)
        except Exception as error:
            test_full_scene_submission_errors[scene_id] = str(error)
            print(f'提交失败 [{test_site_id}, {scene_id}]：{error}', flush=True)

    return {
        'site_id': test_site_id,
        'requested_scenes': len(test_scene_ids),
        'submitted_this_run': submitted_this_run,
        'recorded_tasks': len(test_full_scene_tasks),
        'submission_errors': dict(test_full_scene_submission_errors)
    }

In [11]:
# 短元数据/小块掩膜检查；不提交任务。失败或超时会恢复原超时设置。
ee.data.setDeadline(25_000)
ee.data.setMaxRetries(0)
try:
    mask_source_check = validate_water_mask_source()
    first_scene_check = test_scene_ids[0].strip()
    image_id_check = (first_scene_check if first_scene_check.startswith('COPERNICUS/S1_GRD/')
                      else f'COPERNICUS/S1_GRD/{first_scene_check}')
    image_check = ee.Image(image_id_check)
    time_check = ee.Date(image_check.get('system:time_start'))
    hour_check = ee.Date.fromYMD(time_check.get('year'), time_check.get('month'), time_check.get('day')).advance(time_check.get('hour'), 'hour')
    era5_count_check = (ee.ImageCollection('ECMWF/ERA5/HOURLY')
                        .filterBounds(image_check.geometry())
                        .filterDate(hour_check, hour_check.advance(1, 'hour')).size().getInfo())
    print('MASK_SOURCE:', mask_source_check, flush=True)
    print('SAR_BANDS:', image_check.bandNames().getInfo(), flush=True)
    print('ERA5_HOUR_COUNT:', era5_count_check, flush=True)
    if era5_count_check == 0:
        raise ValueError('首景对应小时没有 ERA5，暂不提交任务。')
finally:
    ee.data.setDeadline(120_000)
    ee.data.setMaxRetries(5)


MASK_SOURCE: {'collection': 'MODIS/006/MOD44W', 'year': 2015, 'count': 1, 'bands': ['water_mask', 'water_mask_QA'], 'mask_probe': {'water_mask': 0}}


SAR_BANDS: ['VV', 'VH', 'angle']


ERA5_HOUR_COUNT: 1


In [12]:
# 手动运行：默认只新增提交 1 个首站点任务，先确认其完成，再继续。
# 已排队、运行或完成的景会跳过；FAILED/CANCELLED 可以重试。
# 新任务前缀 test_fullscene_v8_；不会自动取消旧任务。
submission_report = submit_first_site_sar(max_new_tasks=1)
submission_report


[1/14] 检查影像：COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20150511T224246_20150511T224308_005880_007928_C66D


已提交：test_fullscene_v8_AIAN18_01_20150511_007928_C66D；task_id=OQKHXAR3FUD5H5U5ZZKBMUSO


{'site_id': 'AIAN18',
 'requested_scenes': 14,
 'submitted_this_run': 1,
 'recorded_tasks': 1,
 'submission_errors': {}}

In [ ]:
# 可重复运行查看任务状态；COMPLETED 才表示导出完成。
# 不进行自动轮询，也不重新提交任务。
status_rows = []
for scene_id, task in test_full_scene_tasks.items():
    try:
        status = task.status()
        status_rows.append({
            'site_id': test_site_id,
            'scene_id': scene_id,
            'task_id': task.id,
            'state': status.get('state'),
            'error_message': status.get('error_message', '')
        })
    except Exception as error:
        status_rows.append({
            'site_id': test_site_id,
            'scene_id': scene_id,
            'task_id': task.id,
            'state': 'STATUS_QUERY_ERROR',
            'error_message': str(error)
        })

pd.DataFrame(status_rows, columns=[
    'site_id', 'scene_id', 'task_id', 'state', 'error_message'
])